In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

model = "llama-3.3-70b-versatile"

In [ ]:
SYSTEM_PROMPT = """
You are a helpful personal assistant.
Use your tools when you need real data.
Keep your answers concise.
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_calendar",
            "description": "Check calendar events for a given date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string"
                    }
                },
                "required": ["date"]
            }
        }
    }
]

In [ ]:
def check_calendar(date):
    return "10am: Standup, 2pm: Dentist"

In [ ]:
def run_agent(messages):

    while True:

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason
        msg = response.choices[0].message

        messages.append(msg)

        if finish_reason == "stop":
            return msg.content

        if finish_reason == "tool_calls":

            for tc in msg.tool_calls:

                name = tc.function.name
                args = json.loads(tc.function.arguments)

                if name == "check_calendar":
                    result = check_calendar(**args)
                else:
                    result = f"Unknown tool: {name}"

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })

In [ ]:
questions = [
    "What's on my calendar today?",
    "Tell me about the standup.",
    "What time is my dentist?",
    "Am I free at 3pm?",
    "Summarise my day."
]

In [ ]:
for q in questions:

    messages.append({
        "role": "user",
        "content": q
    })

    answer = run_agent(messages)

    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Messages in history: {len(messages)}")
    print()

In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
def trim_history(messages, max_messages=6):

    if len(messages) > max_messages:
        return [messages[0]] + messages[-(max_messages - 1):]

    return messages

In [ ]:
for q in questions:

    messages.append({
        "role": "user",
        "content": q
    })

    answer = run_agent(messages)

    print(f"Q: {q}")
    print(f"A: {answer}")

    messages = trim_history(
        messages,
        max_messages=6
    )

    print(f"Messages in history: {len(messages)}")
    print()